# Cell Detection using YOLOv8 (n, s, m, l, x) on EL Images

## Project Overview

This notebook trains and evaluates multiple YOLOv8 variants for object detection on Electroluminescence (EL) cell images.

The objective is to compare the performance of different YOLOv8 architectures for cell detection and determine the most suitable model based on detection accuracy, training behavior, and computational cost.

---

## Models Trained

The following YOLOv8 variants are trained:

- YOLOv8n
- YOLOv8s
- YOLOv8m
- YOLOv8l
- YOLOv8x

---

## Dataset Information

**Dataset Source:** Roboflow

**Dataset:** CellDetection

**Task:** Object Detection

**Number of Classes:** 1

Class names:

- `cell`

Dataset structure:

```text
dataset/
│
├── train/
│   ├── images/
│   └── labels/
│
├── valid/
│   ├── images/
│   └── labels/
│
├── test/
│   ├── images/
│   └── labels/
│
└── data.yaml
```

Train, validation, and test splits are automatically generated by Roboflow.

---

## Hardware Configuration

| Component | Specification |
|-----------|---------------|
| CPU | AMD Ryzen Threadripper 7960X |
| RAM | 192 GB |
| GPU | NVIDIA RTX PRO 4500 Blackwell |
| Operating System | Ubuntu 24.04.2 LTS |

---

## Training Configuration

| Parameter | Value |
|------------|--------|
| Image Size | 1024 |
| Epochs | 100 |
| Early Stopping Patience | 15 |
| Seed | 42 |

Batch size varies depending on model size:

| Model | Batch Size |
|--------|------------|
| YOLOv8n | 32 |
| YOLOv8s | 32 |
| YOLOv8m | 24 |
| YOLOv8l | 16 |
| YOLOv8x | 8 |

---

## Data Leakage Prevention Strategy

To avoid data leakage:

- Dataset splitting is completed before training.
- Validation and test datasets remain unchanged.
- Augmentations are applied only during training.
- Validation and test images are never augmented.
- The same preprocessing pipeline is applied across all models.

---

## EL Image Augmentation Strategy

Since Electroluminescence images are grayscale-like images with limited color information, color-based augmentations such as hue and saturation modifications are disabled.

Applied augmentations:

| Augmentation | Value |
|-------------|--------|
| Hue Shift | 0.0 |
| Saturation Shift | 0.0 |
| Brightness Shift | 0.15 |
| Rotation | ±5° |
| Translation | 0.05 |
| Scaling | 0.15 |
| Horizontal Flip | 50% |
| Vertical Flip | Disabled |
| Mosaic | 0.30 |
| MixUp | 0.05 |

Mosaic augmentation is disabled during the final 10 epochs to improve convergence.

---

## Metrics Collected

The following metrics are collected for each model:

- Precision
- Recall
- mAP@0.50
- mAP@0.50:0.95
- Training Loss
- Validation Loss

---

## Saved Outputs

### Saved Models

```text
saved_models/

    yolov8n_best.pt
    yolov8s_best.pt
    yolov8m_best.pt
    yolov8l_best.pt
    yolov8x_best.pt
```

### Saved Metrics

```text
saved_metrics/

    yolov8n.pt_history.csv
    yolov8n.pt_history.json
    yolov8n.pt_metrics.json

    ...

    final_summary.csv
```

### Saved Plots

```text
saved_plots/

    yolov8n_loss.png
    yolov8s_loss.png
    yolov8m_loss.png
    yolov8l_loss.png
    yolov8x_loss.png

    map_comparison.png
    precision_recall.png
```

---

## Generated Visualizations

The notebook automatically generates:

- Training loss curves
- Validation loss curves
- Precision comparison
- Recall comparison
- mAP comparison

Raw metric values are additionally stored in CSV and JSON formats to allow later plotting using custom visualization styles.

---

## Reproducibility

The experiment uses a fixed random seed:

```python
SEED = 42
```

This improves reproducibility of results across training runs.

---

## Experimental Objective

The purpose of this experiment is to compare YOLOv8 architectures based on:

- Detection accuracy
- Precision
- Recall
- mAP performance
- Training behavior
- Model size
- Computational efficiency

and identify the most suitable architecture for EL cell detection.

In [2]:
!pip install ultralytics roboflow pandas matplotlib numpy

In [3]:
from ultralytics import YOLO
from roboflow import Roboflow

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import shutil
import os

from pathlib import Path

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ANA3jCmwICfdHyeh2Rr2")
project = rf.workspace("solarvortex").project("celldetection-willm")
version = project.version(9)
dataset = version.download("yolov8")

DATASET_YAML = f"{dataset.location}/data.yaml"

print(DATASET_YAML)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CellDetection-9 in yolov8:: 100%|█| 8789/8789 


/home/user/SolarVortex/CellExtraction/YOLO/YOLOv8/CellDetection-9/data.yaml


In [4]:
# =========================================================
# YOLOv8 Benchmark Pipeline for PV EL Cell Detection
# =========================================================
#
# Features:
# - Benchmarks YOLOv8s/m/l/x
# - Early stopping
# - Saves best weights
# - Saves all metrics to CSV
# - Saves training curves
# - Saves plots
# - Saves raw graph points for later plotting
# - Saves inference speed
# - Saves params + GFLOPs
# - Saves confusion matrices generated by YOLO
# - Robust path handling
#
# =========================================================

import os
import time
import shutil
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

# =========================================================
# CONFIG
# =========================================================

DATASET_YAML = "CellDetection-9/data.yaml"

MODELS = [
    "yolov8s.pt",
    "yolov8m.pt",
    "yolov8l.pt",
    "yolov8x.pt"
]

EPOCHS = 300
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 30

PROJECT_NAME = "benchmark_results"

# =========================================================
# CREATE OUTPUT DIR
# =========================================================

os.makedirs(PROJECT_NAME, exist_ok=True)

# =========================================================
# STORAGE
# =========================================================

benchmark_results = []

# =========================================================
# TRAIN LOOP
# =========================================================

for model_name in MODELS:

    print("\n================================================")
    print(f"Training {model_name}")
    print("================================================")

    model_id = model_name.replace(".pt", "")

    # =====================================================
    # LOAD MODEL
    # =====================================================

    model = YOLO(model_name)

    # =====================================================
    # TRAIN
    # =====================================================

    start_time = time.time()

    train_results = model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        project=PROJECT_NAME,
        name=model_id,
        pretrained=True,
        optimizer="AdamW",
        lr0=1e-3,
        device=0,
        workers=16,
        cache=True,
        amp=True,
        plots=True,
        save=True,
        save_period=-1,
        verbose=True
    )

    total_training_time = time.time() - start_time

    # =====================================================
    # SAVE DIRECTORY
    # =====================================================

    save_dir = train_results.save_dir

    print(f"\nResults Directory: {save_dir}")

    # =====================================================
    # VALIDATE BEST MODEL
    # =====================================================

    best_model_path = os.path.join(
        save_dir,
        "weights",
        "best.pt"
    )

    best_model = YOLO(best_model_path)

    metrics = best_model.val()

    # =====================================================
    # EXTRACT METRICS
    # =====================================================

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)

    map50 = float(metrics.box.map50)
    map5095 = float(metrics.box.map)

    # =====================================================
    # DERIVED METRICS
    # =====================================================

    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

    # IoU approximation from mAP50
    iou = map50

    # Dice Score
    dsc = (2 * iou) / (1 + iou + 1e-8)

    # =====================================================
    # SPEED METRICS
    # =====================================================

    preprocess_time = metrics.speed["preprocess"]
    inference_time = metrics.speed["inference"]
    postprocess_time = metrics.speed["postprocess"]

    total_latency = (
        preprocess_time +
        inference_time +
        postprocess_time
    )

    fps = 1000 / total_latency

    # =====================================================
    # MODEL INFO
    # =====================================================

    model_info = best_model.info(verbose=False)

    # Manual definitions because info() formatting changes
    params = best_model.model.parameters()

    total_params = sum(p.numel() for p in params)

    # GFLOPs already known from summary output
    # Approximate extraction
    try:
        gflops = best_model.model.info()["GFLOPs"]
    except:
        gflops = None

    # =====================================================
    # STORE RESULTS
    # =====================================================

    benchmark_results.append({

        "Model": model_id,

        "Precision": precision,
        "Recall": recall,
        "F1": f1,

        "mAP50": map50,
        "mAP50-95": map5095,

        "IoU": iou,
        "DSC": dsc,

        "Preprocess(ms)": preprocess_time,
        "Inference(ms)": inference_time,
        "Postprocess(ms)": postprocess_time,

        "TotalLatency(ms)": total_latency,
        "FPS": fps,

        "TrainingTime(sec)": total_training_time,
        "TrainingTime(hr)": total_training_time / 3600,

        "Parameters": total_params,
        "GFLOPs": gflops,

        "BestWeights": best_model_path,
        "ResultsDir": str(save_dir)

    })

    # =====================================================
    # LOAD TRAINING CURVES
    # =====================================================

    results_csv = os.path.join(
        save_dir,
        "results.csv"
    )

    curves_df = pd.read_csv(results_csv)

    # =====================================================
    # SAVE RAW CURVES
    # =====================================================

    curves_csv_path = os.path.join(
        save_dir,
        "training_curves.csv"
    )

    curves_df.to_csv(curves_csv_path, index=False)

    # =====================================================
    # PLOT LOSS CURVES
    # =====================================================

    plt.figure(figsize=(12, 6))

    plt.plot(
        curves_df["epoch"],
        curves_df["train/box_loss"],
        label="Train Box Loss"
    )

    plt.plot(
        curves_df["epoch"],
        curves_df["val/box_loss"],
        label="Val Box Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_id} Box Loss Curve")

    plt.legend()

    plt.grid(True)

    plt.savefig(
        os.path.join(
            save_dir,
            "loss_curve.png"
        )
    )

    plt.close()

    # =====================================================
    # PLOT mAP CURVES
    # =====================================================

    plt.figure(figsize=(12, 6))

    plt.plot(
        curves_df["epoch"],
        curves_df["metrics/mAP50(B)"],
        label="mAP50"
    )

    plt.plot(
        curves_df["epoch"],
        curves_df["metrics/mAP50-95(B)"],
        label="mAP50-95"
    )

    plt.xlabel("Epoch")
    plt.ylabel("mAP")
    plt.title(f"{model_id} mAP Curve")

    plt.legend()

    plt.grid(True)

    plt.savefig(
        os.path.join(
            save_dir,
            "map_curve.png"
        )
    )

    plt.close()

    # =====================================================
    # PLOT PRECISION / RECALL
    # =====================================================

    plt.figure(figsize=(12, 6))

    plt.plot(
        curves_df["epoch"],
        curves_df["metrics/precision(B)"],
        label="Precision"
    )

    plt.plot(
        curves_df["epoch"],
        curves_df["metrics/recall(B)"],
        label="Recall"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{model_id} Precision Recall Curve")

    plt.legend()

    plt.grid(True)

    plt.savefig(
        os.path.join(
            save_dir,
            "precision_recall_curve.png"
        )
    )

    plt.close()

    print(f"\nFinished {model_id}")

# =========================================================
# SAVE BENCHMARK SUMMARY
# =========================================================

benchmark_df = pd.DataFrame(benchmark_results)

summary_csv = os.path.join(
    PROJECT_NAME,
    "benchmark_summary.csv"
)

benchmark_df.to_csv(summary_csv, index=False)

print("\n================================================")
print("BENCHMARK SUMMARY")
print("================================================")

print(benchmark_df)

# =========================================================
# COMPARISON PLOTS
# =========================================================

metrics_to_plot = [

    "mAP50",
    "mAP50-95",

    "Precision",
    "Recall",
    "F1",

    "IoU",
    "DSC",

    "FPS",

    "Inference(ms)",

    "TrainingTime(hr)"

]

# =========================================================
# GENERATE COMPARISON PLOTS
# =========================================================

for metric in metrics_to_plot:

    plt.figure(figsize=(10, 6))

    plt.bar(
        benchmark_df["Model"],
        benchmark_df[metric]
    )

    plt.xlabel("Model")
    plt.ylabel(metric)

    plt.title(f"{metric} Comparison")

    plt.grid(True)

    plt.savefig(
        os.path.join(
            PROJECT_NAME,
            f"{metric}_comparison.png"
        )
    )

    plt.close()

# =========================================================
# SAVE RAW COMPARISON POINTS
# =========================================================

for metric in metrics_to_plot:

    metric_df = benchmark_df[["Model", metric]]

    metric_df.to_csv(
        os.path.join(
            PROJECT_NAME,
            f"{metric}_points.csv"
        ),
        index=False
    )

# =========================================================
# FINAL MESSAGE
# =========================================================

print("\n================================================")
print("ALL BENCHMARKS COMPLETED")
print("================================================")

print(f"\nResults saved in: {PROJECT_NAME}")


Training yolov8s.pt
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.12.0+cu130 CUDA:0 (NVIDIA RTX PRO 4500 Blackwell, 32119MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=CellDetection-9/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s-2, nbs=64, nms=False, opset=None, optimize=False, optimize